# 00 · Dataset Curation & Ground-truth Definition

**Purpose.** Define and count the validation datasets (FMDV, PRRSV, PEDV): how many records, which genes carry truth annotation, and which records are usable ground truth. Produces the numbers behind **Table 1** of the paper.

## Inputs

- FMD: `FMD_100seq_anno.gb` (+ ref `FMD_ref_test.gb`)
- PRRS: `PRRS_100seq_anno.gb` (+ ref `PRRS_ref_test.gb`)
- PED: `PED_100seqs.gb` (+ refs `PED_ref_1.gb`, `PED_ref_2.gb`)

In [1]:
from pathlib import Path
import sys, time

import pandas as pd
import matplotlib.pyplot as plt

# --- anchor ROOT to the repo (folder that contains app/src) ---
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA   = ROOT / "app" / "data"
CONFIG = ROOT / "app" / "config"

# References (verify these are the intended ref records for the paper):
FMD_REF    = DATA / "FMD"  / "FMD_ref_test.gb"        # alt: FMD_FJ175661_Anno.gb
FMD_QUERY  = DATA / "FMD"  / "FMD_100seq_anno.gb"
PRRS_REF   = DATA / "PRRS" / "PRRS_ref_test.gb"       # alt: PRRS_MT746146_Anno.gb
PRRS_QUERY = DATA / "PRRS" / "PRRS_100seq_anno.gb"
PED_REFS   = {"ref_1": DATA / "PED" / "PED_ref_1.gb",
              "ref_2": DATA / "PED" / "PED_ref_2.gb"}
PED_QUERY  = DATA / "PED" / "PED_100seqs.gb"

# Run toggle: keep False for a fast smoke test, True for the full 100-record run.
RUN_FULL = False
SAMPLE_N = 10


In [2]:
# outputs land inside this unit folder so figures/tables sit next to the notebook
UNIT_DIR = ROOT / "app" / "validation" / "00_dataset"
OUT = UNIT_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

outputs -> /Users/MacbookProCuaNminh/Desktop/Nminh-Code/Bio-Vin/Phase_2/viralift/app/validation/00_dataset_curation/outputs


## Run — parse records and count truth features (real modules)

In [3]:
from app.src.io.genbank_parser import load_genbank_records, parse_cds_features, parse_mat_peptides, get_record_metadata
from app.src.alias.gene_alias import apply_alias_to_features
from app.validation._shared.validation_utils import load_reference_bundle

DATASETS = {
    "FMDV":  {"query": FMD_QUERY,  "ref": FMD_REF},
    "PRRSV": {"query": PRRS_QUERY, "ref": PRRS_REF},
    "PEDV":  {"query": PED_QUERY,  "ref": PED_REFS["ref_1"]},
}

rows = []
for virus, cfg in DATASETS.items():
    bundle = load_reference_bundle(cfg["ref"])
    alias_lookup = bundle["alias_lookup"]
    records = load_genbank_records(cfg["query"])
    for rec in records:
        feats = apply_alias_to_features(parse_cds_features(rec), alias_lookup)
        for f in feats:
            rows.append({"virus": virus, "record_id": rec.id,
                         "gene": f.get("name"), "name_source": f.get("name_source")})
truth = pd.DataFrame(rows)
truth.head()

,virus,record_id,gene,name_source
0,FMDV,AY686687.1,polyprotein,excluded
1,FMDV,AY687333.1,polyprotein,excluded
2,FMDV,AY687334.1,polyprotein,excluded
3,FMDV,DQ989303.1,polyprotein,excluded
4,FMDV,DQ989304.1,polyprotein,excluded


## Metrics / Tables

- `#records` per virus, `#gene occurrences`, `#distinct genes`
- gene presence matrix (how many records carry each gene in truth)

In [4]:
dataset_table = (truth.groupby("virus")
                 .agg(records=("record_id", "nunique"),
                      gene_occurrences=("gene", "size"),
                      distinct_genes=("gene", "nunique"))
                 .reset_index())
dataset_table.to_csv(OUT / "dataset_table.tsv", sep="\t", index=False)
gene_presence = (truth.groupby(["virus", "gene"])["record_id"].nunique()
                 .reset_index(name="records_with_truth"))
gene_presence.to_csv(OUT / "gene_presence.tsv", sep="\t", index=False)
dataset_table

,virus,records,gene_occurrences,distinct_genes
0,FMDV,100,100,1
1,PEDV,98,601,40
2,PRRSV,100,836,12


## Interpretation

> ⚠️ **TODO**: 1–2 câu: dataset nào là ground-truth hợp lệ, vì sao cần strict-clean subset (ORF1ab gộp không dùng làm denominator cho ORF1a/1b).